# Our transformer, our recipe, their world

**The cheap half of a 2×2.** [`../othello_transfer/`](../othello_transfer/) put *our probe and our
editor* on *their model* and they worked (null 2.723 → 0.016, Edit Index −0.829 → +0.697). Every
editor we have fails on discworld. This notebook fills the third cell: **our architecture, trained
with our discworld recipe, on their world.**

| | their world (Othello) | our world (discworld) |
|---|---|---|
| **their architecture** — 8 blocks, `d_model` 512, full causal | editable ✓ (2026-08-20/21) | run A — not run |
| **our architecture** — 4 blocks, `d_model` 256, RoPE, banded | **this notebook** | not editable ✗ |

- **Editable here** → our architecture and recipe are exonerated; the discworld negative is about
  the world or our data, and run A becomes largely redundant.
- **Not editable here, with the gates passed** → the difference is architectural, and run A becomes
  the priority with a sharp hypothesis instead of a fishing expedition.
- It says **nothing** about whether discworld is the hard part. Only run A does that.

## The two design choices that make it interpretable

**1. Data scale is a controlled axis, not a free variable.** Training at Li et al.'s 20M games
would be 222× the unique sequences `W16` ever saw, so a success there would tell us nothing about
our setting. Every rung of the ladder therefore runs the **same 95,100 optimiser steps** with the
identical schedule and varies *only* the size of the pool it samples from. Anything that moves
across M → D is data diversity; compute is held fixed.

**2. Every gate is held-out.** Arm `M` sees 90k games 300 times, so training loss reaches wherever
memorisation can take it. All gates are computed at `best_model.pt` on games from a **disjoint
index range**. This mirrors `W16`, which overfits discworld's training loss (best val at epoch ~40
of 300) and is nevertheless a good predictor out of distribution — Sevan, 2026-08-21:
*"discworld is not just memorizing, it does well OOD on the test set."*

Run codes, configurations and every cited number: [`OURS_ON_OTHELLO_RUNS.md`](OURS_ON_OTHELLO_RUNS.md).

## Definitions

Every metric below is **imported**, never re-derived here — `othello_data`, `othello_probe`,
`transfer_pipeline` and `linear_intervention` from [`../othello_transfer/`](../othello_transfer/),
which are the same modules that produced the numbers on Li et al.'s own checkpoint
(`harness/ANALYSIS.md` §1). Restated in full because a deliverable must stand alone
(`harness/STYLE.md` §5).

### The models

| code | descriptive label | detail |
|---|---|---|
| `M_w16` | **our transformer · matched data · window 16** | `d_model` 256, 4 layers, 4 heads, RoPE, band 16 → `state_span` 61, **5 residual points**. `nn.Embedding(61, 256)` in, `Linear(256, 61)` out, cross-entropy |
| `M_w40` | **our transformer · matched data · window 40** | identical but band 40 → `state_span` 157 |
| `random init` | **untrained, same architecture** | Li et al.'s own `--random` control. The comparator every decodability number is read against |

`state_span = n_layers·(window − 1) + 1`, so **window 16 already covers a 60-move game** (61 ≥ 60).
The band costs *directness*, not reach: their model attends from move 59 to any earlier move in one
hop at all 8 layers, ours routes through up to 4 hops. `w40` removes that difference at zero extra
compute (measured 25,018 vs 25,001 games/s).

### The scale ladder — fixed compute, varying diversity

| code | unique games | epochs over its pool | unique tokens | vs `W16`'s 3.6M frames | steps |
|---|---|---|---|---|---|
| `M` | 90,000 | 300 | 5.4M | **1.5× — matched** | 95,100 |
| `L1` | 1,000,000 | 27 | 60M | 17× | 95,100 |
| `L2` | 5,000,000 | 5.4 | 300M | 83× | 95,100 |
| `D` | 20,000,000 | 1.35 | 1.2B | 333× | 95,100 |
| `F` | 20,000,000 | 8 passes | 1.2B | 333× | **8× the steps** |

⚠ **`F` is not a scale datapoint** — it is the only arm with more compute, and answers "can this
architecture do Othello at all", never "did scale do it".

### Held-out gates (test split, disjoint index range, at `best_model.pt`)

| gate | formula | units | better |
|---|---|---|---|
| **legal-move mass** | mean over positions of $\sum_{m \in \text{legal}(t)} p(m \mid \text{history})$ | 0…1 | ↑ |
| **top-1 legal rate** | fraction of positions where $\arg\max p$ is a legal move | 0…1 | ↑ |
| **top-1 accuracy** | fraction where $\arg\max p$ is the move actually played | 0…1 | ↑ |
| **held-out CE** | $-\frac{1}{N}\sum \log p(\text{move}_{t+1})$ over non-pad positions | nats | ↓ |
| ***Bayes top-1*** | $\text{mean}(1/|\text{legal}(t)|)$ | 0…1 | — |
| ***Bayes CE*** | $\text{mean}(\log|\text{legal}(t)|)$ | nats | — |

⚠ **Top-1 accuracy and CE have a hard ceiling that is not 1 and not 0.** Their generator draws
uniformly from the legal set, so a *perfect* model scores exactly the Bayes rate. The quantity that
means something is **excess CE over Bayes CE**. Published reference: Li et al. report **0.9998**
legal-move mass for their 25.3M-parameter model.

### Probes

`othello_probe.fit_probe`, unchanged — linear, `MLP 128 hidden` (theirs), `MLP 512 hidden` (ours);
Adam, 200 epochs, batch 4096, lr 1e-3, 20% held out, inputs standardised inside the probe. One
probe per **(target, family, split, residual point)**, never shared across points.

| axis | levels |
|---|---|
| **target** | `state` = absolute colour (white/blank/black — **theirs**) · `mine` = blank/mine/theirs relative to the player to move (**Nanda's**) |
| **split** | `frame` = pooled rows (**their** convention) · `sequence` = whole games (**our** convention, `harness/ANALYSIS.md` §2) |

Reported as **held-out error rate (%)**, ↓, always beside the **majority-class** floor and the
**random-init** model's error at the same point. Published: Li et al.'s nonlinear probe **1.7%**,
linear **20.4%**.

### Editors — all write to the residual stream at the last position, then let the network recompute

| editor | write | source |
|---|---|---|
| **Layerwise MLP Grad Steering @$L_s$** | $x \leftarrow x - \alpha\,\partial L/\partial x$ at $L_s$ **and every point after it**; $L = $ target term $+\ \beta\times$ hold-the-rest | `othello_probe.make_intervention_hook`, unmodified (Li et al. App. G, $\beta = 0.2$) |
| **Nanda Direction Addition** | $x \leftarrow x + \alpha\,p_d$, $p_d$ = the linear probe's weight column for the target class | `linear_intervention.run(mode="add")` |
| **Nanda, target − current** | $x \leftarrow x + \alpha\,(p_{\text{tgt}} - p_{\text{cur}})$ | same, `subtract=True` |
| **Pseudoinverse Injection** | $\Delta = A^{+}(\text{target} - (Ax + b))$, minimum-norm, null-space preserving | `pim.editors.probe_steering.inject_state`, **unmodified** |

Each is run **at every point at once** and **at each single point**, because 2026-08-21 showed
those differ by 28× on their model.

### Intervention metrics

| metric | definition | units | better |
|---|---|---|---|
| **Li error vs post-flip** | top-*N* predicted moves vs the post-flip legal set, false pos + false neg, $N = |\text{legal}|$ | errors | ↓ |
| **Li error vs pre-flip** | the same against the **pre**-flip set — *the guard* | errors | ↑ for a real edit |
| **Edit Index (union)** | $(d_{\text{uned}} - d_{\text{edit}})/(d_{\text{uned}} + d_{\text{edit}})$, $d$ = RMSE against uniform-over-legal, over the **union** of the two legal sets | −1…+1 | ↑ |
| **Edit Index (symdiff)** | the same over squares whose *legality* changed — a narrower question, never quote it as "the" Edit Index | −1…+1 | ↑ |
| **legal mass** | predicted probability on the post-flip legal set | 0…1 | ↑ |
| **‖Δx‖/‖x‖** | write size relative to the activation — **an axis, not a quality**: it is what makes different α parameterisations comparable | ratio | — |

⚠ **The Edit Index floor is model-dependent and must be recomputed, never borrowed.** On Li et al.'s
model the null intervention scores **−0.829** *because that model is a good predictor of the
unedited world*. A model that predicts nothing scores near **0**. So a higher Edit Index is only
evidence of editing when read against **this** model's own null.

In [ ]:
# [1] Setup. Everything measured here lives in `evaluate.py` / `corpus.py` / `model.py` beside this
#     notebook; every probe, editor and metric is imported from `../othello_transfer/` unchanged.
import json, os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

THREAD = Path.cwd().resolve()
REPO = THREAD.parents[3]
for p in (str(THREAD), str(THREAD.parent / "othello_transfer"),
          str(THREAD.parent / "othello_gpt"), str(REPO), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO)  # `othello_gpt/pipeline` resolves runs/ and datasets/ against the CWD

import corpus as cp
import evaluate as ev
import linear_intervention as li
from pim.figures.theme import PALETTE, style_ax

RUNS = REPO / "runs" / "ours_on_othello"
FIGDIR = RUNS / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
SEED, BETA = 0, 0.2
torch.manual_seed(SEED); np.random.seed(SEED)
RESULTS = {}

# Published / previously established, cited not recomputed.
REF = {
    "li_legal_mass": 0.9998, "li_nonlinear_probe": 1.7, "li_linear_probe": 20.4,
    "li_null": 2.68, "li_best": 0.12, "nanda_add": 0.10,
    # ../othello_transfer/, OUR code on THEIR model
    "xfer_null_li": 2.723, "xfer_null_ei": -0.829,
    "xfer_grad_li": 0.016, "xfer_grad_ei": 0.656,
    "xfer_add_li": 0.062, "xfer_add_ei": 0.603,
    "xfer_pinv_li": 0.052, "xfer_pinv_ei": 0.697,
    # discworld W16, ../transformers/ and ../othello_gpt/
    "w16_unsteered_ei": -0.684, "w16_best_write_ei": -0.194,
    "w16_pos_r2_linear": 0.798, "w16_pos_r2_mlp": 0.9349, "w16_steps": 95_100,
}

# Which runs exist right now — the notebook renders whatever has finished.
AVAIL = sorted(d.name for d in RUNS.iterdir()
               if d.is_dir() and (d / "best_model.pt").exists())
print(f"device {ev.DEV}   ·  checkpoints available: {', '.join(AVAIL) if AVAIL else 'none'}")
for name in AVAIL:
    c = json.loads((RUNS / name / "config.json").read_text())
    print(f"  {name:<8} rung {c['rung']:<3} window {c['model']['window']:<3} "
          f"span {c['state_span']:<4} {c['unique_games']:>10,} unique games  "
          f"{c['total_steps']:>7,} steps ({c['epochs_over_pool']:.2f} epochs)")
print(f"\ndiscworld W16 reference: 90,000 episodes, {REF['w16_steps']:,} steps, best val ~13% in")

In [ ]:
# [2] Held-out gates. Test split, disjoint index range, at `best_model.pt` — never training games.
#     The random-init arm is Li et al.'s own `--random` control and is the floor everything is read
#     against. ~4 min.
t0 = time.time()
paths = cp.build(cp.LADDER["D"], log=lambda s: None, only=("test",))
te_tok, te_len = cp.load(paths["test"])
N_GATE = 2000

MODELS, CKPTS, GATES = {}, {}, {}
for name in AVAIL:
    MODELS[name], CKPTS[name] = ev.load_run(name)
    GATES[name] = ev.gates(MODELS[name], te_tok[:N_GATE], te_len[:N_GATE], log=None)
    print(f"  {name}: legal mass {GATES[name]['legal_mass']:.4f}", flush=True)
if AVAIL:
    rnd = ev.random_init(MODELS[AVAIL[0]])
    MODELS["random init"] = rnd
    GATES["random init"] = ev.gates(rnd, te_tok[:N_GATE], te_len[:N_GATE], log=None)
    print(f"  random init: legal mass {GATES['random init']['legal_mass']:.4f}")
print(f"\n{time.time() - t0:.0f}s  ·  {GATES[AVAIL[0]]['n_positions']:,} held-out positions "
      f"from {N_GATE:,} games")

display(Markdown(
    f"**Table 1 — held-out generalisation gates.** Test split ({N_GATE:,} games from index range "
    f"[{cp.TEST_LO:,}, {cp.TEST_LO + cp.TEST_N:,}), disjoint from training). *Italic* columns are "
    "the ceilings this data imposes, not targets.\n\n" + ev.gate_table(GATES)))

g = GATES[AVAIL[0]]
print(f"\nThe Bayes rate is the ceiling: a PERFECT model scores top-1 {g['bayes_top1']:.4f} and "
      f"CE {g['bayes_ce']:.4f},")
print("because the generator draws uniformly from the legal set. Read excess CE, not CE.")
RESULTS["gates"] = GATES
RESULTS["gate_config"] = {"n_games": N_GATE, "test_lo": cp.TEST_LO}

In [ ]:
# [3] Fig 1 — training curves, with the discworld parallel marked. `W16` bottoms out ~13% into its
#     schedule and rises after; the question is whether ours does the same at matched data.
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6), facecolor="white")
STY = {n: dict(color=PALETTE[i % len(PALETTE)], ls="-" if "w16" in n else "--", lw=2)
       for i, n in enumerate(AVAIL)}
CURVE, TOTAL = {}, {}
for name in AVAIL:
    rows = [json.loads(x) for x in (RUNS / name / "metrics.jsonl").read_text().splitlines()]
    CURVE[name] = rows
    TOTAL[name] = json.loads((RUNS / name / "config.json").read_text())["total_steps"]
    st = [r["step"] for r in rows]
    axes[0].plot(st, [r["val_loss"] for r in rows], label=f"{name} · val", **STY[name])
    axes[0].plot(st, [r["train_loss"] for r in rows], alpha=0.35, lw=1.2,
                 color=STY[name]["color"], ls=":")
    b = min(rows, key=lambda r: r["val_loss"])
    axes[0].plot([b["step"]], [b["val_loss"]], "o", ms=8, mfc="none", mew=2,
                 color=STY[name]["color"])
    axes[1].plot([100 * r["step"] / TOTAL[name] for r in rows],
                 [r["val_loss"] - GATES[name]["bayes_ce"] for r in rows], label=name, **STY[name])
axes[0].axhline(GATES[AVAIL[0]]["bayes_ce"], color="0.3", ls="-.", lw=1.6,
                label=f"Bayes CE ({GATES[AVAIL[0]]['bayes_ce']:.3f}) — the data's own floor")
axes[0].set_xlabel("optimiser step"); axes[0].set_ylabel("cross-entropy (nats)")
axes[0].set_title("(a) train (dotted) and val (solid); ○ = best val")
axes[1].axhline(0, color="0.3", ls="-.", lw=1.6, label="Bayes floor")
axes[1].axvline(13, color="0.6", ls=":", lw=1.8,
                label="discworld W16's best val (~13% through its schedule)")
axes[1].set_xlabel("% through the schedule"); axes[1].set_ylabel("val CE − Bayes CE (nats)")
axes[1].set_title("(b) excess over the floor — the comparable axis")
for ax in axes:
    ax.legend(fontsize=8); style_ax(ax)
fig.suptitle("Fig 1 — training, against the ceiling this data imposes", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93), w_pad=2.0)
fig.savefig(FIGDIR / "fig1_training.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'run':<10}{'best val':>10}{'excess':>9}{'at step':>10}{'of':>9}{'% in':>7}")
for name in AVAIL:
    b = min(CURVE[name], key=lambda r: r["val_loss"])
    print(f"{name:<10}{b['val_loss']:>10.4f}{b['val_loss'] - GATES[name]['bayes_ce']:>9.4f}"
          f"{b['step']:>10,}{TOTAL[name]:>9,}{100 * b['step'] / TOTAL[name]:>6.0f}%")
print("\ndiscworld W16 for comparison: best val at epoch ~40 of 300 = 13% through, then rises.")
RESULTS["curves"] = CURVE

In [ ]:
# [4] The probe grid: 2 targets x 3 families x 2 splits x 5 residual points = 60 probes per
#     checkpoint, each fit independently on its own point. This is `../othello_gpt/`'s probe and
#     its fitting loop, imported unchanged. The random-init arm gets the same grid — it is the
#     comparator, not an afterthought. ~15 min per model.
t0 = time.time()
data = ev.probe_data()
n_rows = int(data.mask.sum())
print(f"probe corpus: {len(data.tokens):,} held-out games, {n_rows:,} (activation, board) rows, "
      f"index range [{cp.PROBE_LO:,}, {cp.PROBE_LO + cp.PROBE_N:,})")

GRIDS = {}
for name, m in MODELS.items():
    ev.attach(m)
    GRIDS[name] = ev.tp.fit_probe_grid(
        m, data, targets=("state", "mine"),
        families=("MLP 512 hidden", "MLP 128 hidden", "linear"),
        splits=("frame", "sequence"), holdout=0.2, epochs=200, batch=4096, lr=1e-3,
        seed=SEED, log=lambda s: None,
    )
    e = [s["error_rate"] for s in GRIDS[name].stats]
    print(f"  {name:<12} {len(GRIDS[name].stats)} probes, error {min(e):.2f}%–{max(e):.2f}%",
          flush=True)
print(f"\n{(time.time() - t0) / 60:.1f} min")
RESULTS["probe_stats"] = {k: v.stats for k, v in GRIDS.items()}

In [ ]:
# [5] Table 2 — probe error by residual point, every family x target x split, with the two
#     baselines that make an absolute error rate mean something: the MAJORITY-CLASS floor and the
#     RANDOM-INIT model at the same point. A trained model that does not beat random init at a
#     point has not learned a board representation there, whatever the absolute number looks like.
NP = ev.tp.N_POINTS
TRAINED = [n for n in MODELS if n != "random init"]


def err(name, tgt, fam, spl, pt):
    for s in GRIDS[name].stats:
        if (s["target"], s["family"], s["split"], s["point"]) == (tgt, fam, spl, pt):
            return s["error_rate"]
    return float("nan")


def maj(tgt, spl, pt):
    for s in GRIDS[TRAINED[0]].stats:
        if (s["target"], s["split"], s["point"]) == (tgt, spl, pt):
            return s["majority_class_error_rate"]
    return float("nan")


for spl in ("sequence", "frame"):
    rows = ["| target · family · model | " + " | ".join(f"pt {p}" for p in range(NP)) + " | best |",
            "|---|" + "---|" * (NP + 1)]
    for tgt in ("state", "mine"):
        rows.append(f"| *{tgt} — majority-class floor* | "
                    + " | ".join(f"*{maj(tgt, spl, p):.1f}*" for p in range(NP)) + " | — |")
        for fam in ("linear", "MLP 128 hidden", "MLP 512 hidden"):
            for name in list(MODELS):
                v = [err(name, tgt, fam, spl, p) for p in range(NP)]
                tag = "**" if name != "random init" else ""
                rows.append(f"| {tgt} · {fam} · {name} | "
                            + " | ".join(f"{tag}{x:.2f}{tag}" for x in v)
                            + f" | {min(v):.2f} |")
    display(Markdown(
        f"**Table 2{'a' if spl == 'sequence' else 'b'} — held-out probe error rate (%), split by "
        f"{spl.upper()}.** Lower is better. *Italic* rows are the majority-class floor. Published: "
        f"Li et al.'s nonlinear probe {REF['li_nonlinear_probe']}%, linear "
        f"{REF['li_linear_probe']}%.\n\n" + "\n".join(rows)))

print(f"{'model':<14}{'best linear':>12}{'best MLP512':>12}{'majority':>10}   (target=mine, "
      f"split=sequence)")
for name in list(MODELS):
    bl = min(err(name, "mine", "linear", "sequence", p) for p in range(NP))
    bm = min(err(name, "mine", "MLP 512 hidden", "sequence", p) for p in range(NP))
    print(f"{name:<14}{bl:>12.2f}{bm:>12.2f}{min(maj('mine', 'sequence', p) for p in range(NP)):>10.2f}")
print("\n⚠ MLP >= linear tripwire: a strictly more expressive probe scoring WORSE than a linear one")
print("   is a training failure, never a fact about the representation (caught two real bugs here).")
bad = [(n, tg, sp, pt) for n in MODELS for tg in ("state", "mine")
       for sp in ("frame", "sequence") for pt in range(NP)
       if err(n, tg, "MLP 512 hidden", sp, pt) > err(n, tg, "linear", sp, pt) + 1.0]
print(f"   violations with a >1pp gap: {len(bad)}" + (f"  {bad[:6]}" if bad else "  — none"))

In [ ]:
# [6] Fig 2 — decodability by residual point, trained against random init on the same axis. The
#     LEVEL is not the evidence; the GAP to random init is, and so is the depth trend (2026-08-21:
#     on discworld random decodability falls with depth while trained rises).
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), facecolor="white")
pts = list(range(NP))
for ax, tgt in zip(axes, ("mine", "state")):
    for i, name in enumerate(list(MODELS)):
        rnd = name == "random init"
        for fam, mk in (("linear", "s"), ("MLP 512 hidden", "o")):
            ax.plot(pts, [err(name, tgt, fam, "sequence", p) for p in pts],
                    color=PALETTE[i % len(PALETTE)], ls=":" if rnd else ("--" if fam == "linear" else "-"),
                    marker=mk, ms=5, lw=1.4 if rnd else 2, alpha=0.8 if rnd else 1.0,
                    label=f"{name} · {'linear' if fam == 'linear' else 'MLP 512'}")
    ax.plot(pts, [maj(tgt, "sequence", p) for p in pts], color="0.35", ls="-.", lw=1.8,
            label="majority-class floor")
    ax.axhline(REF["li_nonlinear_probe"], color="0.6", ls=":", lw=1.5,
               label=f"Li et al. nonlinear ({REF['li_nonlinear_probe']}%)")
    ax.axhline(REF["li_linear_probe"], color="0.8", ls=":", lw=1.5,
               label=f"Li et al. linear ({REF['li_linear_probe']}%)")
    ax.set_xticks(pts); ax.set_xlabel("residual point (0 = embedding)")
    ax.set_ylabel("held-out probe error rate (%)  — lower is better")
    ax.set_title(f"target: {tgt}" + (" (mine/theirs — Nanda's)" if tgt == "mine"
                                     else " (absolute colour — Li's)"))
    ax.legend(fontsize=7.5, ncol=2); style_ax(ax)
fig.suptitle("Fig 2 — board-state decodability, held out by SEQUENCE", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93), w_pad=2.0)
fig.savefig(FIGDIR / "fig2_decodability.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [7] The null arm, then every editor at EVERY residual point at once. The null must be recomputed
#     per model — on Li et al.'s checkpoint it is -0.829 *because that model predicts the unedited
#     world well*; a model that predicts nothing scores near 0, and borrowing their floor would
#     manufacture an edit out of thin air. ~10 min.
t0 = time.time()
bench = ev.od.load_benchmark()
cur_lab, tgt_lab = ev.case_targets(bench)
A_ADD = (0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35, 0.5, 0.75, 1.0)
A_PINV = (0.05, 0.12, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0)
A_GRAD = (0.005, 0.01, 0.02, 0.05, 0.1)

UNS, SWEEP = {}, {}
for name, m in MODELS.items():
    ev.attach(m); li.N_POINTS = ev.tp.N_POINTS
    UNS[name] = ev.unsteered(m, bench)
    lin = ev.linear_probes(GRIDS[name], "mine", "frame")
    mlp = ev.mlp_probes(GRIDS[name], "state", "frame", "MLP 512 hidden")
    allp = set(range(ev.tp.N_POINTS))
    for tag, mode, alphas, sub in (
        ("Nanda: add target direction", "add", A_ADD, False),
        ("Nanda: add (target - current)", "add", A_ADD, True),
        ("OURS: pseudoinverse injection", "pinv", A_PINV, False),
    ):
        for a in alphas:
            _, c = li.run(m, bench, lin, mode, a, allp, tgt_lab, cur_lab, sub)
            SWEEP[(name, tag, a)] = c
    for a in A_GRAD:                       # the gradient editor, at each start layer
        for ls in range(ev.tp.N_POINTS):
            pr, rec = ev.tp.run_arm(m, bench, mlp, ls, alpha=a, n_steps=100, beta=BETA)
            c = ev.od.scorecard(pr, bench)
            c["write_ratio"] = float(np.mean([v for k, v in rec.items()
                                              if "write_ratio" in str(k)] or [np.nan]))
            SWEEP[(name, f"Li grad steering @L_s={ls}", a)] = c
    print(f"  {name}: null Li {UNS[name]['li_error_vs_post']:.3f} "
          f"EI {UNS[name]['edit_index_union']:+.3f}", flush=True)
print(f"\n{(time.time() - t0) / 60:.1f} min")
RESULTS["unsteered"] = {k: {a: b for a, b in v.items() if not a.endswith("per_case")}
                        for k, v in UNS.items()}
RESULTS["sweep_all_points"] = {f"{n}|{t}|{a}": {k: v for k, v in c.items()
                                                if not k.endswith("per_case")}
                               for (n, t, a), c in SWEEP.items()}

In [ ]:
# [8] Table 3 — the best arm per editor per model, against that model's OWN null. Selection is on
#     Li error, which selects on an outcome metric: admissible only because the question here is
#     "what is this mechanism's ceiling on this model", not "does it work". The guard column and
#     the write size are reported so a destroyed model cannot masquerade as an edit.
def best_arms(name):
    tags = sorted({t for (n, t, a) in SWEEP if n == name})
    out = {}
    for t in tags:
        ks = [(n, tt, a) for (n, tt, a) in SWEEP if n == name and tt == t]
        out[t] = min(ks, key=lambda k: SWEEP[k]["li_error_vs_post"])
    return out


hdr = ("| model · editor | best α | ‖Δx‖/‖x‖ | Li err vs post ↓ | Li err vs pre ↑ | "
       "Edit Index union ↑ | symdiff ↑ | legal mass ↑ |")
rows = [hdr, "|---|" + "---|" * 7]
for name in MODELS:
    u = UNS[name]
    rows.append(f"| **{name} — no intervention** | — | — | {u['li_error_vs_post']:.3f} | "
                f"{u['li_error_vs_pre']:.3f} | **{u['edit_index_union']:+.3f}** | "
                f"{u['edit_index_symdiff']:+.3f} | {u['legal_mass']:.3f} |")
    for t, k in best_arms(name).items():
        c = SWEEP[k]
        rows.append(f"| {name} · {t} | {k[2]} | {c.get('write_ratio', float('nan')):.3f} | "
                    f"**{c['li_error_vs_post']:.3f}** | {c['li_error_vs_pre']:.3f} | "
                    f"**{c['edit_index_union']:+.3f}** | {c['edit_index_symdiff']:+.3f} | "
                    f"{c['legal_mass']:.3f} |")
rows.append("| *Li et al., their model, their code* | — | — | *0.12* | — | — | — | — |")
rows.append("| *Nanda et al., their model* | — | — | *0.10* | — | — | — | — |")
rows.append(f"| *OUR code on THEIR model — pseudoinverse, 1 point* | — | — "
            f"| *{REF['xfer_pinv_li']}* | — | *{REF['xfer_pinv_ei']:+.3f}* | — | — |")
rows.append(f"| *OUR code on THEIR model — null* | — | — | *{REF['xfer_null_li']}* | — "
            f"| *{REF['xfer_null_ei']:+.3f}* | — | — |")
display(Markdown("**Table 3 — every editor written at ALL residual points, best α per arm.**\n\n"
                 + "\n".join(rows)))

print("Gain over each model's OWN null (the only comparison that means anything):")
for name in MODELS:
    u = UNS[name]["edit_index_union"]
    g = {t: SWEEP[k]["edit_index_union"] - u for t, k in best_arms(name).items()}
    bt = max(g, key=g.get)
    print(f"  {name:<14} null {u:+.3f}  best {bt} → {u + g[bt]:+.3f}  (gain {g[bt]:+.3f})")
print(f"\nOn THEIR model the same code moved {REF['xfer_null_ei']:+.3f} → {REF['xfer_pinv_ei']:+.3f} "
      f"(gain {REF['xfer_pinv_ei'] - REF['xfer_null_ei']:+.3f}).")
print(f"On discworld W16 it moved {REF['w16_unsteered_ei']:+.3f} → {REF['w16_best_write_ei']:+.3f} "
      f"(gain {REF['w16_best_write_ei'] - REF['w16_unsteered_ei']:+.3f}).")

In [ ]:
# [9] SINGLE-POINT writes. 2026-08-21 found this matters by 28x on their model: a recomputed
#     injection re-imposes "hold everything else" against the already-edited stream and undoes
#     itself, while a fixed direction cannot. Both mechanisms, every point, best α per cell. ~5 min.
t0 = time.time()
SINGLE = {}
for name, m in MODELS.items():
    ev.attach(m); li.N_POINTS = ev.tp.N_POINTS
    lin = ev.linear_probes(GRIDS[name], "mine", "frame")
    for tag, mode, alphas in (("OURS pseudoinverse", "pinv", A_PINV),
                              ("Nanda addition", "add", A_ADD)):
        for ell in range(ev.tp.N_POINTS):
            bc, ba = None, None
            for a in alphas:
                _, c = li.run(m, bench, lin, mode, a, {ell}, tgt_lab, cur_lab)
                if bc is None or c["li_error_vs_post"] < bc["li_error_vs_post"]:
                    bc, ba = c, a
            SINGLE[(name, tag, ell)] = {"alpha": ba,
                                        **{k: v for k, v in bc.items()
                                           if not k.endswith("per_case")}}
    print(f"  {name} done", flush=True)
print(f"{(time.time() - t0) / 60:.1f} min\n")

for name in MODELS:
    rows = ["| point written | " + " | ".join(str(p) for p in range(NP)) + " | **all** |",
            "|---|" + "---|" * (NP + 1)]
    for tag, alltag in (("OURS pseudoinverse", "OURS: pseudoinverse injection"),
                        ("Nanda addition", "Nanda: add target direction")):
        ka = best_arms(name)[alltag]
        rows.append(f"| {tag} — Li error ↓ | "
                    + " | ".join(f"{SINGLE[(name, tag, p)]['li_error_vs_post']:.3f}"
                                 for p in range(NP))
                    + f" | **{SWEEP[ka]['li_error_vs_post']:.3f}** |")
        rows.append(f"| {tag} — Edit Index | "
                    + " | ".join(f"{SINGLE[(name, tag, p)]['edit_index_union']:+.3f}"
                                 for p in range(NP))
                    + f" | **{SWEEP[ka]['edit_index_union']:+.3f}** |")
    display(Markdown(f"**Table 4 — {name}: one residual point vs all of them.** Best α per cell. "
                     f"Its own null is Edit Index {UNS[name]['edit_index_union']:+.3f}.\n\n"
                     + "\n".join(rows)))
RESULTS["single_point"] = {f"{n}|{t}|{p}": v for (n, t, p), v in SINGLE.items()}

In [ ]:
# [10] Fig 3 — the interventions. (a) absolute Li error against the write size, the axis that makes
#      different α parameterisations comparable. (b) Edit Index gain over each model's OWN null,
#      which is the only cross-model comparison that means anything.
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), facecolor="white")
TAGS = ["Nanda: add target direction", "Nanda: add (target - current)",
        "OURS: pseudoinverse injection"]
MK = {"Nanda: add target direction": "o", "Nanda: add (target - current)": "^",
      "OURS: pseudoinverse injection": "s"}
for i, name in enumerate(MODELS):
    for t in TAGS:
        ks = sorted([k for k in SWEEP if k[0] == name and k[1] == t], key=lambda k: k[2])
        if not ks:
            continue
        axes[0].plot([SWEEP[k]["write_ratio"] for k in ks],
                     [SWEEP[k]["li_error_vs_post"] for k in ks],
                     color=PALETTE[i % len(PALETTE)], marker=MK[t], ms=5, lw=1.8,
                     ls={"o": "-", "^": "-.", "s": "--"}[MK[t]],
                     alpha=0.55 if name == "random init" else 1.0,
                     label=f"{name} · {t.split(':')[-1].strip()}")
    axes[0].axhline(UNS[name]["li_error_vs_post"], color=PALETTE[i % len(PALETTE)], ls=":", lw=1.2,
                    alpha=0.6)
axes[0].axhline(REF["li_best"], color="0.5", ls="--", lw=1.5, label=f"Li et al. ({REF['li_best']})")
axes[0].axhline(REF["xfer_pinv_li"], color="0.75", ls="-.", lw=1.5,
                label=f"our code on THEIR model ({REF['xfer_pinv_li']})")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("write size ‖Δx‖/‖x‖"); axes[0].set_ylabel("Li error vs post-flip (log) ↓")
axes[0].set_title("(a) all points at once — dotted = each model's own null")

pts = list(range(NP))
for i, name in enumerate(MODELS):
    u = UNS[name]["edit_index_union"]
    for tag, mk, ls in (("OURS pseudoinverse", "s", "--"), ("Nanda addition", "o", "-")):
        axes[1].plot(pts, [SINGLE[(name, tag, p)]["edit_index_union"] - u for p in pts],
                     color=PALETTE[i % len(PALETTE)], marker=mk, ls=ls, ms=5, lw=1.8,
                     alpha=0.55 if name == "random init" else 1.0, label=f"{name} · {tag}")
axes[1].axhline(0, color="0.3", ls="-", lw=1.5, label="each model's own null (no edit)")
axes[1].axhline(REF["xfer_pinv_ei"] - REF["xfer_null_ei"], color="0.5", ls="--", lw=1.5,
                label=f"our code on THEIR model (+{REF['xfer_pinv_ei'] - REF['xfer_null_ei']:.3f})")
axes[1].axhline(REF["w16_best_write_ei"] - REF["w16_unsteered_ei"], color="0.75", ls="-.", lw=1.5,
                label=f"best write on discworld W16 "
                      f"(+{REF['w16_best_write_ei'] - REF['w16_unsteered_ei']:.3f})")
axes[1].set_xticks(pts); axes[1].set_xlabel("the single residual point written to")
axes[1].set_ylabel("Edit Index GAIN over this model's own null ↑")
axes[1].set_title("(b) one point at a time")
for ax in axes:
    ax.legend(fontsize=7, ncol=1); style_ax(ax)
fig.suptitle("Fig 3 — probe-derived writes on our transformer, trained on Othello", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93), w_pad=2.0)
fig.savefig(FIGDIR / "fig3_interventions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [11] Table 5 — the 2x2 this thread exists to fill, all four cells on one page. Every number in
#      the other three cells is CITED from its own notebook, never recomputed here.
best_name = min(TRAINED, key=lambda n: min(
    SWEEP[k]["li_error_vs_post"] for k in SWEEP if k[0] == n)) if TRAINED else None
gain = {}
for n in MODELS:
    u = UNS[n]["edit_index_union"]
    gain[n] = max(SWEEP[k]["edit_index_union"] - u for k in SWEEP if k[0] == n)

rows = [
    "| architecture | world | probe: best board error | null Edit Index | best Edit Index | gain | source |",
    "|---|---|---|---|---|---|---|",
    f"| **theirs** (8 blk, d512, full causal) | **theirs** | {REF['li_nonlinear_probe']}% "
    f"(their nonlinear probe) | {REF['xfer_null_ei']:+.3f} | {REF['xfer_pinv_ei']:+.3f} | "
    f"**{REF['xfer_pinv_ei'] - REF['xfer_null_ei']:+.3f}** | `../othello_transfer/` |",
]
for n in TRAINED:
    be = min(min(err(n, t, f, "sequence", p) for p in range(NP))
             for t in ("state", "mine") for f in ("MLP 512 hidden", "linear"))
    u = UNS[n]["edit_index_union"]
    rows.append(f"| **ours** (4 blk, d256, {'band 16' if 'w16' in n else 'band 40'}) | **theirs** "
                f"| {be:.2f}% | {u:+.3f} | {u + gain[n]:+.3f} | **{gain[n]:+.3f}** | "
                f"this notebook (`{n}`) |")
u = UNS["random init"]["edit_index_union"]
be = min(min(err("random init", t, f, "sequence", p) for p in range(NP))
         for t in ("state", "mine") for f in ("MLP 512 hidden", "linear"))
rows.append(f"| *ours, UNTRAINED* | *theirs* | *{be:.2f}%* | *{u:+.3f}* | *{u + gain['random init']:+.3f}* "
            f"| *{gain['random init']:+.3f}* | *this notebook — the control* |")
rows.append(f"| **ours** (4 blk, d256, band 16) | **ours** (discworld) | "
            f"R² {REF['w16_pos_r2_mlp']} (position, not an error rate) | "
            f"{REF['w16_unsteered_ei']:+.3f} | {REF['w16_best_write_ei']:+.3f} | "
            f"**{REF['w16_best_write_ei'] - REF['w16_unsteered_ei']:+.3f}** | "
            f"`../transformers/`, `../othello_gpt/` |")
rows.append("| **theirs** | **ours** (discworld) | — | — | — | — | "
            "run A — `directions/othello-architecture-on-discworld.md`, not run |")
display(Markdown("**Table 5 — the 2×2.** ⚠ The probe column is not comparable across worlds "
                 "(a 3-class board error rate vs a continuous position R²); the **gain** column "
                 "is, because each is read against its own null.\n\n" + "\n".join(rows)))
RESULTS["two_by_two_gain"] = gain

In [ ]:
# [12] Summary — rendered from the values computed above rather than transcribed, so it cannot go
#      stale against the tables.
def fmt(x, n=3):
    return f"{x:+.{n}f}"


lines = ["## What this run establishes", ""]
g0 = GATES[TRAINED[0]]
lines += [
    "### 1 · Did our architecture learn Othello?", "",
    "| model | legal-move mass | excess CE over Bayes | best val at |",
    "|---|---|---|---|",
]
for n in TRAINED:
    b = min(CURVE[n], key=lambda r: r["val_loss"])
    lines.append(f"| {n} | **{GATES[n]['legal_mass']:.4f}** | "
                 f"{GATES[n]['ce'] - GATES[n]['bayes_ce']:+.4f} nats | "
                 f"step {b['step']:,} of {TOTAL[n]:,} ({100 * b['step'] / TOTAL[n]:.0f}%) |")
lines.append(f"| *random init* | *{GATES['random init']['legal_mass']:.4f}* | "
             f"*{GATES['random init']['ce'] - GATES['random init']['bayes_ce']:+.4f}* | — |")
lines.append(f"| *Li et al., 25.3M params, 20M games* | *{REF['li_legal_mass']}* | — | — |")
lines += ["", f"The Bayes-optimal predictor on this data scores top-1 {g0['bayes_top1']:.4f} and "
              f"CE {g0['bayes_ce']:.4f} — the generator draws uniformly from the legal set, so "
              "**excess CE is the quantity, not CE**.", ""]

lines += ["### 2 · Is the board state decodable, above the random-init floor?", ""]
for n in TRAINED:
    bl = min(err(n, "mine", "linear", "sequence", p) for p in range(NP))
    bm = min(err(n, "mine", "MLP 512 hidden", "sequence", p) for p in range(NP))
    rl = min(err("random init", "mine", "linear", "sequence", p) for p in range(NP))
    rm = min(err("random init", "mine", "MLP 512 hidden", "sequence", p) for p in range(NP))
    mj = min(maj("mine", "sequence", p) for p in range(NP))
    lines.append(f"- **{n}** (mine/theirs, held out by sequence): linear **{bl:.2f}%** vs random "
                 f"init {rl:.2f}%, MLP **{bm:.2f}%** vs random init {rm:.2f}%; majority floor "
                 f"{mj:.2f}%. Training buys **{rl - bl:+.2f}pp** linear / **{rm - bm:+.2f}pp** MLP "
                 f"over an untrained network of the same architecture.")
lines += ["", f"Li et al. report {REF['li_nonlinear_probe']}% nonlinear / {REF['li_linear_probe']}% "
              "linear on a model with 8× the parameters and 222× the data. The comparison that "
              "licenses anything downstream is not to them — it is to **random init at the same "
              "point**.", ""]

lines += ["### 3 · Is it editable?", ""]
lines += ["| model | own null | best editor | best Edit Index | gain |", "|---|---|---|---|---|"]
for n in list(MODELS):
    u = UNS[n]["edit_index_union"]
    bt = max(best_arms(n), key=lambda t: SWEEP[best_arms(n)[t]]["edit_index_union"])
    bv = SWEEP[best_arms(n)[bt]]["edit_index_union"]
    sp = max(((t, p) for (nn, t, p) in SINGLE if nn == n),
             key=lambda tp: SINGLE[(n, tp[0], tp[1])]["edit_index_union"])
    spv = SINGLE[(n, sp[0], sp[1])]["edit_index_union"]
    which = f"{bt} (all points)" if bv >= spv else f"{sp[0]} @ point {sp[1]}"
    best = max(bv, spv)
    em = "*" if n == "random init" else "**"
    lines.append(f"| {em}{n}{em} | {fmt(u)} | {which} | {fmt(best)} | {em}{fmt(best - u)}{em} |")
lines += [
    f"| *our code on THEIR model* | *{fmt(REF['xfer_null_ei'])}* | *pseudoinverse, 1 point* | "
    f"*{fmt(REF['xfer_pinv_ei'])}* | *{fmt(REF['xfer_pinv_ei'] - REF['xfer_null_ei'])}* |",
    f"| *the same code on discworld W16* | *{fmt(REF['w16_unsteered_ei'])}* | *best of all* | "
    f"*{fmt(REF['w16_best_write_ei'])}* | *{fmt(REF['w16_best_write_ei'] - REF['w16_unsteered_ei'])}* |",
    "",
    "⚠ **Read the gain column, never the level.** The null Edit Index is a property of how well a "
    "model predicts the *unedited* world, so a weaker predictor starts closer to 0 and an edit that "
    "does nothing can look better than one that works. The random-init row is in the table for "
    "exactly this reason.",
    "",
    "### 4 · What this does and does not settle", "",
    "This is one rung — `M`, matched to discworld's data volume. It bounds what our architecture "
    "does at **our** data scale, in a world where their architecture is known to be editable. The "
    "remaining rungs (`L1`, `L2`, `D` at fixed compute, then `F`) separate *architecture* from "
    "*data diversity*; until they land, a weak result here is consistent with either.",
]
display(Markdown("\n".join(lines)))

In [ ]:
# [13] Serialise. Its own file — `runs/ours_on_othello/results.json`. Per-case arrays are dropped
#      here but the full sweeps are kept: a filter that silently drops the curves a plot depends on
#      has bitten this repo before (`harness/ANALYSIS.md` §1), so what is dropped is named.
out = RUNS / "results.json"
RESULTS["config"] = {
    "seed": SEED, "beta": BETA, "n_gate_games": N_GATE, "probe_games": len(data.tokens),
    "probe_rows": n_rows, "alphas": {"add": list(A_ADD), "pinv": list(A_PINV),
                                     "grad": list(A_GRAD)},
    "n_points": NP, "runs": AVAIL, "ladder": cp.LADDER,
    "seed_ranges": {"train": [cp.TRAIN_LO, cp.LADDER["D"]],
                    "test": [cp.TEST_LO, cp.TEST_LO + cp.TEST_N],
                    "probe": [cp.PROBE_LO, cp.PROBE_LO + cp.PROBE_N]},
    "reference_values": REF,
    "dropped": "per-case arrays (li_error_per_case, edit_index_per_case)",
}


def _plain(o):
    if isinstance(o, dict):
        return {str(k): _plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_plain(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


out.write_text(json.dumps(_plain(RESULTS), indent=1))
print(f"wrote {out.relative_to(REPO)}  ({out.stat().st_size:,} bytes)")
print("figures: " + ", ".join(sorted(p.name for p in FIGDIR.glob("fig*.png"))))